In [1]:
# %%capture --no-stderr
# !pip install -U langgraph
# !pip install psycopg2-binary==2.9.9
# !pip install python-dotenv

In [2]:
from pprint import pprint

In [3]:
import os
import sys
import boto3
import json
import logging
import mlflow
from mlflow.entities import SpanType


from dotenv import load_dotenv
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [4]:
mlflow.__version__

'2.20.3'

In [5]:
# Get the absolute path to the backend directory
current_dir = os.getcwd()  # Get current working directory
backend_path = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(backend_path)

# Load environment variables from .env file
load_dotenv()

True

In [6]:
from backend import app

In [7]:
# # Enable auto-tracing for Amazon Bedrock
mlflow.bedrock.autolog()
mlflow.langchain.autolog()

2025/03/25 18:02:43 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.


In [8]:
try:
    mlflow_arn = os.getenv('MLFLOW_TRACKING_ARN')
    if mlflow_arn:
        print(f'\nMLFlow tracking URI: {mlflow_arn}')
        mlflow.set_tracking_uri(mlflow_arn)
        print('Successfully connected to MLFlow tracking server')
except Exception as e:
    print('Error connecting to MLFlow tracking server')
    raise e


MLFlow tracking URI: arn:aws:sagemaker:us-west-2:577201992296:mlflow-tracking-server/bedrock-chatbot-mlflow
Successfully connected to MLFlow tracking server


In [9]:
import uuid
session_id = str(uuid.uuid4())

In [10]:
session_id

'e54958db-fe11-4ec2-bc00-d1e1397828bc'

In [11]:
mlflow.set_experiment(str(session_id))
mlflow.tracing.enable_notebook_display()

2025/03/25 18:02:45 INFO mlflow.tracking.fluent: Experiment with name 'e54958db-fe11-4ec2-bc00-d1e1397828bc' does not exist. Creating a new experiment.


In [12]:
messages = 'Hello, I need some help'
# messages = "I need help with an order. I don't remember my order id. My email is john@example.com"

In [13]:
session_dict = {'session_id': session_id}

In [14]:
response = app.execute_workflow(
    task=messages,
    session_dict=session_dict
)
response.get('response')


 <============== Node: entry_intent ===================> 



"<reply>Hello! Welcome to AnyCompany customer support. I'm here to help you look up your account, check orders, or assist with order cancellations. How may I assist you today?</reply>"

In [15]:
messages = "I don't remember my order id"
response = app.execute_workflow(
    task=messages,
    session_dict=session_dict
)
response.get('response')


 <============== Node: entry_intent ===================> 



"<reply>I can help you find your order information. To look up your orders, I'll need some information to identify your account first. Could you please provide either your:\n- Email address\n- Phone number\n- or Username\n\nOnce you provide any of these details, I can help you locate your order history.</reply>"

In [16]:
messages = "My email is anacarolina_silva@example.com"
response = app.execute_workflow(
    task=messages,
    session_dict=session_dict
)
response.get('response')


 <============== Node: entry_intent ===================> 


 <=============== Node: order_confirmation =============> 


 ------- RETRY 1 with delay 1.09s ---------- 



'<reply>Hello Ana Carolina, I found your recent orders:\n\n1. Order #86420: Fitness Tracker (Status: Delivered)\n2. Order #81695: Wireless Keyboard (Status: Processing)\n\nWhich order would you like to know more about?</reply>'

In [17]:
messages = "I want to cancel the order 81695"
response = app.execute_workflow(
    task=messages,
    session_dict=session_dict
)
response.get('response')


 <=============== Node: order_confirmation =============> 


 ------- RETRY 1 with delay 1.06s ---------- 


 <=============== Node: resolution =====================> 


 ------- CANCEL ORDER: 81695 ---------- 


 ------- QUERY: UPDATE orders SET status = 'Cancelled' WHERE id = %s AND status = 'Processing' ----------- 

Cancel successful: Mockup function not real implementation

 ------- RETRY 1 with delay 1.04s ---------- 



"<reply>I've successfully cancelled your order #81695 for the Wireless Keyboard. You should receive a confirmation email shortly. Is there anything else I can help you with?</reply>"

In [18]:
messages = "I'm all set. Thank you"
response = app.execute_workflow(
    task=messages,
    session_dict=session_dict
)
response.get('response')


 <=============== Node: resolution =====================> 


 ------- RETRY 1 with delay 1.06s ---------- 



"<reply>You're welcome! Thank you for contacting AnyCompany customer support. Have a great day!</reply>"